# Chapter 7 Student Colab Notebook: Eigenvalues and Eigenvectors

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 7.1-7.4  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 7. It contains detailed theory, complete proofs, original worked examples, every intermediate eigenspace and diagonalization calculation, geometric interpretation, applications, Python visualizations, section worksheets, and complete instructor solutions.

> **How to use this student notebook**
>
> Run the cells from top to bottom in Google Colab. Read the detailed theory and proofs, study the worked examples, explore the visualizations, and complete each section worksheet. Complete instructor solution sections are not included.

## Chapter map

1. **Section 7.1:** Eigenvalues, eigenvectors, eigenspaces, and characteristic equations  
2. **Section 7.2:** Similarity and diagonalization  
3. **Section 7.3:** Symmetric matrices and orthogonal diagonalization  
4. **Section 7.4:** Population models, systems of differential equations, quadratic forms, principal axes, and constrained optimization

Each section concludes with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 7
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown
from mpl_toolkits.mplot3d import Axes3D

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

sp.init_printing()

def show_matrix(M, label=None):
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M

def show_vector(v, label=None):
    v = sp.Matrix(v)
    if label:
        display(Math(rf"{label}={sp.latex(v)}"))
    else:
        display(v)
    return v

def show_exact(expr, label):
    expr = sp.simplify(expr)
    display(Math(rf"{label}={sp.latex(expr)}"))
    return expr

def eigenspace_data(A):
    A=sp.Matrix(A)
    data=[]
    for lam,mult,basis in A.eigenvects():
        data.append({'eigenvalue':lam,'algebraic_multiplicity':mult,'basis':basis})
    return data

def plot_eigen_directions_2d(A, title='Invariant directions under a matrix'):
    A=np.asarray(A,dtype=float)
    vals,vecs=np.linalg.eig(A)
    theta=np.linspace(0,2*np.pi,400)
    circle=np.vstack([np.cos(theta),np.sin(theta)])
    transformed=A@circle
    fig,ax=plt.subplots(figsize=(7,6))
    ax.plot(circle[0],circle[1],label='unit circle')
    ax.plot(transformed[0],transformed[1],label='transformed circle')
    for j in range(vecs.shape[1]):
        v=np.real(vecs[:,j]); v=v/np.linalg.norm(v)
        ax.plot([-2*v[0],2*v[0]],[-2*v[1],2*v[1]],linestyle='--',label=f'eigendirection {j+1}')
    ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8)
    ax.set_aspect('equal'); ax.grid(alpha=.3); ax.legend(); ax.set_title(title)
    plt.show()

def plot_population(history, labels):
    hist=np.asarray(history,dtype=float)
    fig,ax=plt.subplots(figsize=(8,5))
    for j,label in enumerate(labels):
        ax.plot(range(len(hist)),hist[:,j],marker='o',label=label)
    ax.set_xlabel('year'); ax.set_ylabel('population index')
    ax.grid(alpha=.3); ax.legend(); ax.set_title('Age-class population dynamics')
    plt.show()

def plot_quadratic_contours(A, level=12, title='Quadratic-form level set'):
    A=np.asarray(A,dtype=float)
    x=np.linspace(-5,5,500); y=np.linspace(-5,5,500)
    X,Y=np.meshgrid(x,y)
    Z=A[0,0]*X**2+2*A[0,1]*X*Y+A[1,1]*Y**2
    fig,ax=plt.subplots(figsize=(7,6))
    ax.contour(X,Y,Z,levels=[level],linewidths=2)
    vals,vecs=np.linalg.eigh(A)
    for j in range(2):
        v=vecs[:,j]
        ax.plot([-4*v[0],4*v[0]],[-4*v[1],4*v[1]],linestyle='--',label=f'principal axis {j+1}')
    ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8)
    ax.set_aspect('equal'); ax.grid(alpha=.3); ax.legend(); ax.set_title(title)
    plt.show()

> **Original-example policy**
>
> All numerical examples are original. Themes are used selectively from the San Antonio Spurs, Backstreet Boys, hiking, badminton, driving, travel, and R&B music. Fictional populations, listening profiles, performance data, and routes are labeled as teaching models.

# Section 7.1 - Eigenvalues and Eigenvectors

## Learning objectives

Students should be able to:

- explain the geometric meaning of an eigenvector;
- verify an eigenvalue-eigenvector pair;
- prove that an eigenspace is a subspace;
- derive and use the characteristic equation;
- compute eigenvalues and bases for eigenspaces;
- distinguish algebraic multiplicity from geometric multiplicity;
- find eigenvalues of triangular matrices;
- interpret eigenvalues and eigenvectors of a linear transformation.

### Definition: eigenvalue and eigenvector

> Let $A$ be an $n\times n$ matrix. A scalar $\lambda$ is an eigenvalue of $A$ when there is a nonzero vector $\mathbf v$ such that $A\mathbf v=\lambda\mathbf v$. The nonzero vector $\mathbf v$ is an eigenvector corresponding to $\lambda$.

## Why eigenvectors are special

A general vector may change both direction and length under multiplication by $A$. An eigenvector changes only by a scalar factor:

$$
A\mathbf v=\lambda\mathbf v.
$$

- If $\lambda>1$, the vector is stretched.
- If $0<\lambda<1$, it is contracted.
- If $\lambda<0$, its direction reverses and its length is scaled by $|\lambda|$.
- If $\lambda=0$, the vector is sent to the zero vector.

The line through an eigenvector is therefore an invariant direction of the transformation.

### Why $\mathbf v\ne\mathbf0$ and where $A-\lambda I$ comes from

An eigenvector satisfies

$$
A\mathbf v=\lambda\mathbf v,
\qquad
\mathbf v\ne\mathbf0.
$$

The zero vector is excluded because it satisfies the equation for every $\lambda$ and identifies no special direction.

Rearrange:

$$
A\mathbf v-\lambda\mathbf v
=
(A-\lambda I)\mathbf v
=
\mathbf0.
$$

Thus the eigenvectors for a fixed $\lambda$ are the nonzero vectors in $\ker(A-\lambda I)$. The eigenspace includes those vectors together with $\mathbf0$ so that it forms a subspace.


In [ ]:
A=np.array([[3,1],[1,3]],dtype=float)
plot_eigen_directions_2d(A,'Travel-route transformation: invariant directions')

> **Spurs-inspired verification example**
>
> Consider the fictional matrix $A=\begin{bmatrix}20&1\\1&20\end{bmatrix}$. The numbers are selected from familiar Spurs jersey values; this is not a real dataset.

Let

$$
A=
\begin{bmatrix}
20&1\\
1&20
\end{bmatrix},
\qquad
\mathbf v_+=
\begin{bmatrix}1\\1\end{bmatrix},
\qquad
\mathbf v_-=
\begin{bmatrix}1\\-1\end{bmatrix}.
$$

Then

$$
A\mathbf v_+
=
\begin{bmatrix}21\\21\end{bmatrix}
=21\mathbf v_+,
$$

so $\mathbf v_+$ is an eigenvector with eigenvalue $21$. Similarly,

$$
A\mathbf v_-
=
\begin{bmatrix}19\\-19\end{bmatrix}
=19\mathbf v_-.
$$

The two eigendirections are the lines $y=x$ and $y=-x$.

In [ ]:
A_spurs=sp.Matrix([[20,1],[1,20]])
v_plus=sp.Matrix([1,1]); v_minus=sp.Matrix([1,-1])
show_vector(A_spurs*v_plus,r'A\mathbf v_+')
show_vector(A_spurs*v_minus,r'A\mathbf v_-')

### Definition: eigenspace

> For an eigenvalue $\lambda$, the eigenspace is $E_\lambda=\ker(A-\lambda I)$. It contains all eigenvectors corresponding to $\lambda$ together with the zero vector.

### Theorem: an eigenspace is a subspace

> If $\lambda$ is an eigenvalue of $A$, then $E_\lambda=\ker(A-\lambda I)$ is a subspace of $\mathbb R^n$.

### Complete proof

We verify the subspace conditions.

1. **The zero vector belongs to the set.**

$$
(A-\lambda I)\mathbf 0=\mathbf 0,
$$

so $\mathbf 0\in E_\lambda$.

2. **Closure under addition.** Suppose $\mathbf u,\mathbf v\in E_\lambda$. Then

$$
(A-\lambda I)\mathbf u=\mathbf 0,
\qquad
(A-\lambda I)\mathbf v=\mathbf 0.
$$

Using distributivity,

$$
\begin{aligned}
(A-\lambda I)(\mathbf u+\mathbf v)
&=(A-\lambda I)\mathbf u+(A-\lambda I)\mathbf v\\
&=\mathbf 0+\mathbf 0\\
&=\mathbf 0.
\end{aligned}
$$

Therefore, $\mathbf u+\mathbf v\in E_\lambda$.

3. **Closure under scalar multiplication.** Suppose $\mathbf u\in E_\lambda$ and $c$ is a scalar. Then

$$
(A-\lambda I)(c\mathbf u)
=c(A-\lambda I)\mathbf u
=c\mathbf 0
=\mathbf 0.
$$

Thus $c\mathbf u\in E_\lambda$. All subspace conditions hold, so $E_\lambda$ is a subspace. $\square$

> **Characteristic-equation criterion**
>
> A scalar $\lambda$ is an eigenvalue of $A$ if and only if $\det(A-\lambda I)=0$. Equivalently, one may use $\det(\lambda I-A)=0$; the roots are the same.

### Derivation and proof

Starting from the eigenvalue equation,

$$
A\mathbf v=\lambda\mathbf v,
$$

move all terms to one side:

$$
(A-\lambda I)\mathbf v=\mathbf 0.
$$

We require a **nonzero** solution $\mathbf v$. A homogeneous system $M\mathbf v=\mathbf 0$ has a nonzero solution exactly when $M$ is singular. A square matrix is singular exactly when its determinant is zero. Therefore,

$$
(A-\lambda I)\mathbf v=\mathbf 0
\text{ has a nonzero solution}
\quad\Longleftrightarrow\quad
\det(A-\lambda I)=0.
$$

This determinant is a polynomial in $\lambda$, called the characteristic polynomial. Its roots are the eigenvalues.

## Detailed $2\times2$ eigenvalue calculation

Let

$$
A=
\begin{bmatrix}
4&2\\
1&3
\end{bmatrix}.
$$

The characteristic polynomial is

$$
\begin{aligned}
p(\lambda)
&=\det(A-\lambda I)\\
&=\det
\begin{bmatrix}
4-\lambda&2\\
1&3-\lambda
\end{bmatrix}\\
&=(4-\lambda)(3-\lambda)-2\\
&=\lambda^2-7\lambda+10\\
&=(\lambda-5)(\lambda-2).
\end{aligned}
$$

Hence the eigenvalues are $\lambda=5$ and $\lambda=2$.

### Eigenspace for $\lambda=5$

Solve $(A-5I)\mathbf v=\mathbf 0$:

$$
A-5I=
\begin{bmatrix}
-1&2\\
1&-2
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&-2\\
0&0
\end{bmatrix}.
$$

The equation is

$$
x-2y=0,
$$

so $x=2y$. Let $y=t$. Then

$$
\mathbf v=t
\begin{bmatrix}2\\1\end{bmatrix},
$$

and

$$
E_5=\operatorname{span}\left\{
\begin{bmatrix}2\\1\end{bmatrix}
\right\}.
$$

### Eigenspace for $\lambda=2$

Solve $(A-2I)\mathbf v=\mathbf 0$:

$$
A-2I=
\begin{bmatrix}
2&2\\
1&1
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&1\\
0&0
\end{bmatrix}.
$$

Thus $x+y=0$, so $x=-y$. Let $y=t$. Then

$$
\mathbf v=t
\begin{bmatrix}-1\\1\end{bmatrix},
$$

and

$$
E_2=\operatorname{span}\left\{
\begin{bmatrix}-1\\1\end{bmatrix}
\right\}.
$$

In [ ]:
A=sp.Matrix([[4,2],[1,3]])
lam=sp.symbols('lambda')
show_exact(sp.factor(A.charpoly(lam).as_expr()),r'p(\lambda)')
for ev,mult,basis in A.eigenvects():
    print('eigenvalue =',ev,'algebraic multiplicity =',mult)
    print('basis =',basis)

## Algebraic and geometric multiplicity

The **algebraic multiplicity** of an eigenvalue is its multiplicity as a root of the characteristic polynomial. The **geometric multiplicity** is the dimension of its eigenspace.

Consider

$$
B=
\begin{bmatrix}
2&0&1\\
0&2&0\\
0&0&2
\end{bmatrix}.
$$

Because $B$ is triangular, the characteristic polynomial is

$$
(2-\lambda)^3.
$$

Thus $\lambda=2$ has algebraic multiplicity $3$. However,

$$
B-2I=
\begin{bmatrix}
0&0&1\\
0&0&0\\
0&0&0
\end{bmatrix},
$$

so $x_3=0$ while $x_1$ and $x_2$ are free. Therefore,

$$
E_2=\operatorname{span}\{(1,0,0),(0,1,0)\},
$$

and the geometric multiplicity is $2$.

The **algebraic multiplicity** counts how many times $\lambda$ occurs as a root of the characteristic polynomial. The **geometric multiplicity** is

$$
\dim\ker(A-\lambda I).
$$

Always,

$$
1\le\text{geometric multiplicity}\le\text{algebraic multiplicity}.
$$


### Theorem: eigenvalues of a triangular matrix

> The eigenvalues of a triangular matrix are its diagonal entries, counted with algebraic multiplicity.

### Proof

For an upper triangular matrix $A$, the matrix $A-\lambda I$ is also upper triangular. Its diagonal entries are

$$
a_{11}-\lambda,\ a_{22}-\lambda,\ \ldots,\ a_{nn}-\lambda.
$$

The determinant of a triangular matrix is the product of its diagonal entries. Hence

$$
\det(A-\lambda I)
=\prod_{j=1}^n(a_{jj}-\lambda).
$$

This product equals zero exactly when $\lambda=a_{jj}$ for at least one $j$. Therefore the eigenvalues are the diagonal entries. The same argument applies to lower triangular matrices. $\square$

## Eigenvalues of a linear transformation on $P_2$

Define

$$
T:P_2\to P_2,
\qquad
T(p)=p+xp'.
$$

For $p(x)=a+bx+cx^2$,

$$
p'(x)=b+2cx,
$$

so

$$
T(p)=a+2bx+3cx^2.
$$

Relative to the standard basis $\{1,x,x^2\}$, the matrix is

$$
[T]=
\begin{bmatrix}
1&0&0\\
0&2&0\\
0&0&3
\end{bmatrix}.
$$

Therefore, the eigenvalues are $1,2,3$, with eigenspaces

$$
E_1=\operatorname{span}\{1\},
\qquad
E_2=\operatorname{span}\{x\},
\qquad
E_3=\operatorname{span}\{x^2\}.
$$

> **Useful consequence**
>
> $0$ is an eigenvalue of $A$ if and only if $A$ is singular. Indeed, $A\mathbf v=0\mathbf v$ for some nonzero $\mathbf v$ exactly when $A\mathbf v=\mathbf 0$ has a nontrivial solution.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Eigenvalue | Scalar $\lambda$ for which $A\mathbf v=\lambda\mathbf v$ has a nonzero solution. |
| Eigenvector | Nonzero vector satisfying the eigenvalue equation. |
| Eigenspace | $E_\lambda=\ker(A-\lambda I)$. |
| Characteristic polynomial | $p(\lambda)=\det(A-\lambda I)$. |
| Characteristic equation | $\det(A-\lambda I)=0$. |
| Algebraic multiplicity | Multiplicity of an eigenvalue as a polynomial root. |
| Geometric multiplicity | Dimension of the eigenspace. |
| Invariant direction | A line mapped into itself by the transformation. |

## Section 7.1 worksheet

1. Verify that $(1,1)$ and $(1,-1)$ are eigenvectors of $\begin{bmatrix}20&1\\1&20\end{bmatrix}$ and state their eigenvalues.
2. Find the characteristic polynomial and eigenvalues of $\begin{bmatrix}4&2\\1&3\end{bmatrix}$.
3. Find a basis for each eigenspace in Problem 2.
4. For $B=\begin{bmatrix}2&0&1\\0&2&0\\0&0&2\end{bmatrix}$, find the algebraic and geometric multiplicity of $\lambda=2$.
5. Prove directly that $E_\lambda$ is closed under addition.
6. Find the eigenvalues of the triangular matrix $\begin{bmatrix}9&2&-1\\0&20&4\\0&0&21\end{bmatrix}$.
7. For $T(p)=p+xp'$ on $P_2$, find $T(2-3x+4x^2)$ and identify an eigenbasis.

# Section 7.2 - Diagonalization

## Learning objectives

Students should be able to:

- explain similarity and diagonalization;
- prove that similar matrices have the same characteristic polynomial;
- prove the diagonalization criterion;
- prove that eigenvectors associated with distinct eigenvalues are independent;
- diagonalize a matrix when possible;
- determine why a matrix fails to be diagonalizable;
- use diagonalization to compute powers and simplify a linear transformation.

### Definition: similarity and diagonalization

> Matrices $A$ and $B$ are similar if $B=P^{-1}AP$ for some invertible $P$. A matrix $A$ is diagonalizable if it is similar to a diagonal matrix $D$.

## Why diagonalization is useful

If

$$
A=PDP^{-1},
$$

then

$$
A^k=(PDP^{-1})^k=PD^kP^{-1}.
$$

All middle factors $P^{-1}P$ cancel. Since powers of a diagonal matrix are computed entry by entry, repeated applications of $A$ become much easier.

### Theorem: similar matrices have the same characteristic polynomial

> If $B=P^{-1}AP$, then $\det(\lambda I-B)=\det(\lambda I-A)$. Therefore $A$ and $B$ have the same eigenvalues with the same algebraic multiplicities.

### Complete proof

Assume $B=P^{-1}AP$. Because $P^{-1}IP=I$,

$$
\begin{aligned}
\lambda I-B
&=\lambda P^{-1}IP-P^{-1}AP\\
&=P^{-1}(\lambda I-A)P.
\end{aligned}
$$

Taking determinants and using $\det(XY)=\det(X)\det(Y)$,

$$
\begin{aligned}
\det(\lambda I-B)
&=\det(P^{-1})\det(\lambda I-A)\det(P)\\
&=\frac{1}{\det(P)}\det(\lambda I-A)\det(P)\\
&=\det(\lambda I-A).
\end{aligned}
$$

Thus the characteristic polynomials are identical, so their roots and multiplicities are identical. $\square$

### Theorem: diagonalization criterion

> An $n\times n$ matrix $A$ is diagonalizable if and only if $A$ has $n$ linearly independent eigenvectors.

### Proof: independent eigenvectors imply diagonalization

Suppose $A$ has linearly independent eigenvectors $\mathbf p_1,\ldots,\mathbf p_n$ with corresponding eigenvalues $\lambda_1,\ldots,\lambda_n$. Form

$$
P=[\mathbf p_1\ \mathbf p_2\ \cdots\ \mathbf p_n],
\qquad
D=\operatorname{diag}(\lambda_1,\ldots,\lambda_n).
$$

Because the columns of $P$ are linearly independent, $P$ is invertible. Moreover,

$$
AP=[A\mathbf p_1\ \cdots\ A\mathbf p_n]
=[\lambda_1\mathbf p_1\ \cdots\ \lambda_n\mathbf p_n]
=PD.
$$

Multiplying on the left by $P^{-1}$ gives

$$
P^{-1}AP=D.
$$

Therefore, $A$ is diagonalizable.

### Proof: diagonalization implies independent eigenvectors

Suppose $P^{-1}AP=D$, where $D$ is diagonal and $P$ is invertible. Rearranging gives

$$
AP=PD.
$$

Write the columns of $P$ as $\mathbf p_1,\ldots,\mathbf p_n$, and let the diagonal entries of $D$ be $\lambda_1,\ldots,\lambda_n$. Comparing columns in $AP=PD$ yields

$$
A\mathbf p_j=\lambda_j\mathbf p_j
$$

for every $j$. Thus every column of $P$ is an eigenvector. Because $P$ is invertible, its columns are linearly independent. Hence $A$ has $n$ linearly independent eigenvectors. $\square$

### Theorem: distinct eigenvalues give independent eigenvectors

> Eigenvectors corresponding to distinct eigenvalues are linearly independent.

### Proof by induction

The statement is immediate for one eigenvector. Assume it is true for $k-1$ eigenvectors. Let

$$
A\mathbf v_j=\lambda_j\mathbf v_j,
\qquad
\lambda_1,\ldots,\lambda_k
\text{ distinct},
$$

and suppose

$$
c_1\mathbf v_1+\cdots+c_k\mathbf v_k=\mathbf0.
$$

Apply $A$:

$$
c_1\lambda_1\mathbf v_1+\cdots+c_k\lambda_k\mathbf v_k=\mathbf0.
$$

Multiply the original relation by $\lambda_k$ and subtract:

$$
c_1(\lambda_1-\lambda_k)\mathbf v_1+\cdots+c_{k-1}(\lambda_{k-1}-\lambda_k)\mathbf v_{k-1}=\mathbf0.
$$

By the induction hypothesis, $\mathbf v_1,\ldots,\mathbf v_{k-1}$ are independent. Since every $\lambda_j-\lambda_k\neq0$, we obtain $c_1=\cdots=c_{k-1}=0$. The original relation then gives $c_k\mathbf v_k=0$, and because $\mathbf v_k\neq0$, $c_k=0$. Thus all coefficients are zero. $\square$

## Complete $3\times3$ diagonalization

Consider

$$
A=
\begin{bmatrix}
2&-1&1\\
-1&2&-1\\
1&1&2
\end{bmatrix}.
$$

The characteristic polynomial factors as

$$
\det(\lambda I-A)=(\lambda-1)(\lambda-2)(\lambda-3).
$$

The eigenvalues are distinct, so the corresponding eigenvectors will automatically be linearly independent. We still compute each eigenspace explicitly.

### Eigenvalue $\lambda=1$

$$
A-I=
\begin{bmatrix}
1&-1&1\\
-1&1&-1\\
1&1&1
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&0&1\\
0&1&0\\
0&0&0
\end{bmatrix}.
$$

Thus

$$
x_1+x_3=0,
\qquad
x_2=0.
$$

Let $x_3=t$. Then

$$
\mathbf x=t(-1,0,1),
$$

so choose

$$
\mathbf p_1=(1,0,-1)
$$

as an equivalent eigenvector.

### Eigenvalue $\lambda=2$

$$
A-2I=
\begin{bmatrix}
0&-1&1\\
-1&0&-1\\
1&1&0
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&0&1\\
0&1&-1\\
0&0&0
\end{bmatrix}.
$$

Hence

$$
x_1=-x_3,
\qquad
x_2=x_3.
$$

Let $x_3=t$. Then

$$
\mathbf x=t(-1,1,1),
$$

so choose

$$
\mathbf p_2=(-1,1,1).
$$

### Eigenvalue $\lambda=3$

$$
A-3I=
\begin{bmatrix}
-1&-1&1\\
-1&-1&-1\\
1&1&-1
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&1&0\\
0&0&1\\
0&0&0
\end{bmatrix}.
$$

Thus

$$
x_1=-x_2,
\qquad
x_3=0.
$$

Let $x_2=t$. Then

$$
\mathbf x=t(-1,1,0),
$$

so choose

$$
\mathbf p_3=(-1,1,0).
$$

Form

$$
P=
\begin{bmatrix}
1&-1&-1\\
0&1&1\\
-1&1&0
\end{bmatrix},
\qquad
D=
\begin{bmatrix}
1&0&0\\
0&2&0\\
0&0&3
\end{bmatrix}.
$$

Because $\det(P)=-1$, the matrix $P$ is invertible. Its inverse is

$$
P^{-1}=
\begin{bmatrix}
1&1&0\\
1&1&1\\
-1&0&-1
\end{bmatrix}.
$$

Direct multiplication gives

$$
P^{-1}AP=D.
$$

In [ ]:
A_diag=sp.Matrix([[2,-1,1],[-1,2,-1],[1,1,2]])
P=sp.Matrix([[1,-1,-1],[0,1,1],[-1,1,0]])
D=sp.diag(1,2,3)
show_exact(sp.factor(A_diag.charpoly().as_expr()),r'p(\lambda)')
show_matrix(P,'P'); show_exact(P.det(),r'\det(P)')
show_matrix(P.inv(),'P^{-1}')
show_matrix(P.inv()*A_diag*P,'P^{-1}AP')

## Computing powers by diagonalization

Because $A=PDP^{-1}$,

$$
A^5=PD^5P^{-1},
$$

where

$$
D^5=\operatorname{diag}(1,32,243).
$$

The result is

$$
A^5=
\begin{bmatrix}
212&-31&211\\
-211&32&-211\\
31&31&32
\end{bmatrix}.
$$

This avoids four full matrix multiplications.

If

$$
A=PDP^{-1},
$$

then

$$
A^5=PD^5P^{-1},
$$

and more generally $A^k=PD^kP^{-1}$.


In [ ]:
show_matrix(A_diag**5,'A^5')

> **A matrix that is not diagonalizable**
>
> The matrix $J=\begin{bmatrix}2&1\\0&2\end{bmatrix}$ has only one independent eigenvector, so it is not diagonalizable.

Indeed, the only eigenvalue is $\lambda=2$, with algebraic multiplicity $2$. But

$$
J-2I=
\begin{bmatrix}
0&1\\
0&0
\end{bmatrix},
$$

so $y=0$ and

$$
E_2=\operatorname{span}\{(1,0)\}.
$$

The geometric multiplicity is $1$, smaller than the algebraic multiplicity $2$. Therefore, there are not two linearly independent eigenvectors.

## Diagonalization of a linear transformation

If a linear transformation $T:V\to V$ has a basis

$$
B=\{\mathbf p_1,\ldots,\mathbf p_n\}
$$

consisting of eigenvectors, then the matrix of $T$ relative to $B$ is diagonal:

$$
[T]_B=\operatorname{diag}(\lambda_1,\ldots,\lambda_n).
$$

The diagonal entries record the independent scaling of each eigenvector direction.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Similar matrices | $B=P^{-1}AP$ for an invertible $P$. |
| Diagonalizable | Similar to a diagonal matrix. |
| Eigenbasis | Basis consisting entirely of eigenvectors. |
| Diagonalization | Factorization $A=PDP^{-1}$. |
| Defective matrix | Matrix lacking enough independent eigenvectors. |
| Algebraic multiplicity | Multiplicity in the characteristic polynomial. |
| Geometric multiplicity | Dimension of the eigenspace. |

## Section 7.2 worksheet

1. Prove that similar matrices have the same determinant.
2. Diagonalize $A=\begin{bmatrix}2&-1&1\\-1&2&-1\\1&1&2\end{bmatrix}$ using the eigenvectors supplied in the notes.
3. Verify directly that $AP=PD$ for the matrices in Problem 2.
4. Use diagonalization to compute $A^4$.
5. Determine whether $J=\begin{bmatrix}2&1\\0&2\end{bmatrix}$ is diagonalizable, and justify your answer using eigenspace dimension.
6. Prove that eigenvectors associated with two distinct eigenvalues are linearly independent.
7. Explain why a $3\times3$ matrix with three distinct eigenvalues must be diagonalizable.

# Section 7.3 - Symmetric Matrices and Orthogonal Diagonalization

## Learning objectives

Students should be able to:

- recognize symmetric and orthogonal matrices;
- prove the column criterion for orthogonal matrices;
- prove that eigenvectors of a symmetric matrix associated with distinct eigenvalues are orthogonal;
- understand why the eigenvalues of a real symmetric matrix are real;
- apply the spectral theorem;
- orthogonally diagonalize symmetric matrices, including repeated-eigenvalue cases.

### Definition: symmetric matrix

> A real square matrix is symmetric when $A^T=A$. Thus $a_{ij}=a_{ji}$ for all entries.

### Definition: orthogonal matrix

> A real square matrix $P$ is orthogonal when $P^TP=I$. Equivalently, $P^{-1}=P^T$.

### Theorem: column criterion for orthogonality

> An $n\times n$ matrix $P$ is orthogonal if and only if its columns form an orthonormal set.

### Complete proof

Write

$$
P=[\mathbf p_1\ \cdots\ \mathbf p_n].
$$

The $(i,j)$ entry of $P^TP$ is

$$
(P^TP)_{ij}=\mathbf p_i^T\mathbf p_j=\mathbf p_i\cdot\mathbf p_j.
$$

Therefore,

$$
P^TP=I
$$

exactly when

$$
\mathbf p_i\cdot\mathbf p_j=
\begin{cases}
1,&i=j,\\
0,&i\ne j.
\end{cases}
$$

These are precisely the conditions that the columns form an orthonormal set. $\square$

### Theorem: distinct eigenvalues of a symmetric matrix have orthogonal eigenvectors

> If $A=A^T$, $A\mathbf x=\lambda\mathbf x$, $A\mathbf y=\mu\mathbf y$, and $\lambda\ne\mu$, then $\mathbf x\cdot\mathbf y=0$.

### Complete proof

Because $A$ is symmetric,

$$
\mathbf x^TA\mathbf y=(A\mathbf x)^T\mathbf y.
$$

Using the eigenvalue equations,

$$
\mathbf x^TA\mathbf y
=\mathbf x^T(\mu\mathbf y)
=\mu\mathbf x^T\mathbf y,
$$

while

$$
(A\mathbf x)^T\mathbf y
=(\lambda\mathbf x)^T\mathbf y
=\lambda\mathbf x^T\mathbf y.
$$

Thus

$$
\mu\mathbf x^T\mathbf y=\lambda\mathbf x^T\mathbf y,
$$

so

$$
(\lambda-\mu)\mathbf x^T\mathbf y=0.
$$

Since $\lambda\ne\mu$, it follows that

$$
\mathbf x^T\mathbf y=0.
$$

Therefore the eigenvectors are orthogonal. $\square$

> **Why symmetric matrices have real eigenvalues**
>
> Every eigenvalue of a real symmetric matrix is real.

### Detailed proof using complex vectors

A real matrix can have complex eigenvalues, so temporarily allow a complex eigenvector $\mathbf z\ne0$ with

$$
A\mathbf z=\lambda\mathbf z.
$$

Let $\mathbf z^*$ denote the conjugate transpose. Multiply by $\mathbf z^*$:

$$
\mathbf z^*A\mathbf z=\lambda\mathbf z^*\mathbf z.
$$

The scalar $\mathbf z^*\mathbf z$ is positive and real. We now show that $\mathbf z^*A\mathbf z$ is real. Taking its complex conjugate gives

$$
\overline{\mathbf z^*A\mathbf z}
=\mathbf z^*A^T\mathbf z.
$$

Because $A^T=A$,

$$
\overline{\mathbf z^*A\mathbf z}
=\mathbf z^*A\mathbf z.
$$

A scalar equal to its own complex conjugate is real. Therefore,

$$
\lambda
=\frac{\mathbf z^*A\mathbf z}{\mathbf z^*\mathbf z}
$$

is real. $\square$

### Spectral theorem

> A real matrix is orthogonally diagonalizable if and only if it is symmetric. Thus, for symmetric $A$, there exists an orthogonal $P$ and a real diagonal $D$ such that $A=PDP^T$.

### Proof framework

The full result can be organized in two directions.

**If $A$ is orthogonally diagonalizable, then $A$ is symmetric.** Suppose

$$
A=PDP^T,
$$

where $P$ is orthogonal and $D$ is diagonal. Then

$$
A^T=(PDP^T)^T=PD^TP^T=PDP^T=A,
$$

because $D^T=D$.

**If $A$ is symmetric, then $A$ is orthogonally diagonalizable.** The proof proceeds by induction on the dimension.

1. A symmetric matrix has a real eigenvalue and a corresponding unit eigenvector $\mathbf q_1$.
2. The orthogonal complement $\mathbf q_1^\perp$ is invariant under $A$. To see this, let $\mathbf y\perp\mathbf q_1$. Then

$$
\mathbf q_1^TA\mathbf y
=(A\mathbf q_1)^T\mathbf y
=(\lambda_1\mathbf q_1)^T\mathbf y
=\lambda_1\mathbf q_1^T\mathbf y
=0.
$$

Hence $A\mathbf y\in\mathbf q_1^\perp$.
3. The restriction of $A$ to $\mathbf q_1^\perp$ is again symmetric. By the induction hypothesis, it has an orthonormal eigenbasis.
4. Adjoin $\mathbf q_1$ to that basis. The result is an orthonormal eigenbasis for the entire space.

Using these eigenvectors as columns of $P$ produces $P^TAP=D$. $\square$

## Detailed orthogonal diagonalization

Let

$$
A=
\begin{bmatrix}
5&2\\
2&2
\end{bmatrix}.
$$

The characteristic polynomial is

$$
\begin{aligned}
p(\lambda)
&=\det(A-\lambda I)\\
&=(5-\lambda)(2-\lambda)-4\\
&=\lambda^2-7\lambda+6\\
&=(\lambda-6)(\lambda-1).
\end{aligned}
$$

Thus the eigenvalues are $6$ and $1$.

Along an eigendirection, a unit eigenvector is stretched or reflected only by the scalar $\lambda$, so its image has length $|\lambda|$.


### Eigenvector for $\lambda=6$

$$
A-6I=
\begin{bmatrix}
-1&2\\
2&-4
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
1&-2\\
0&0
\end{bmatrix}.
$$

Thus $x=2y$. Choose

$$
\mathbf v_1=(2,1).
$$

Its norm is $\sqrt5$, so

$$
\mathbf q_1=\frac1{\sqrt5}(2,1).
$$

### Eigenvector for $\lambda=1$

$$
A-I=
\begin{bmatrix}
4&2\\
2&1
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
2&1\\
0&0
\end{bmatrix}.
$$

Thus $2x+y=0$. Choose

$$
\mathbf v_2=(-1,2).
$$

Its norm is $\sqrt5$, so

$$
\mathbf q_2=\frac1{\sqrt5}(-1,2).
$$

Notice that

$$
\mathbf v_1\cdot\mathbf v_2=2(-1)+1(2)=0,
$$

as predicted by the symmetric-matrix theorem.

Form

$$
P=
\frac1{\sqrt5}
\begin{bmatrix}
2&-1\\
1&2
\end{bmatrix},
\qquad
D=
\begin{bmatrix}
6&0\\
0&1
\end{bmatrix}.
$$

The columns of $P$ are orthonormal, so

$$
P^{-1}=P^T.
$$

Direct multiplication verifies

$$
P^TAP=D.
$$

In [ ]:
A_sym=sp.Matrix([[5,2],[2,2]])
q1=sp.Matrix([sp.Rational(2,1)/sp.sqrt(5),sp.Rational(1,1)/sp.sqrt(5)])
q2=sp.Matrix([-sp.Rational(1,1)/sp.sqrt(5),sp.Rational(2,1)/sp.sqrt(5)])
P_sym=sp.Matrix.hstack(q1,q2)
show_matrix(P_sym,'P')
show_matrix(P_sym.T*P_sym,'P^TP')
show_matrix(sp.simplify(P_sym.T*A_sym*P_sym),'P^TAP')
plot_eigen_directions_2d(np.array(A_sym.tolist(),dtype=float),'Symmetric matrix: orthogonal eigendirections')

## Repeated eigenvalue and Gram-Schmidt

Consider

$$
C=
\begin{bmatrix}
2&1&1\\
1&2&1\\
1&1&2
\end{bmatrix}.
$$

The eigenvalues are $4$ and $1$, where $1$ has algebraic multiplicity $2$.

For $\lambda=4$, choose

$$
\mathbf v_3=(1,1,1),
\qquad
\mathbf q_3=\frac1{\sqrt3}(1,1,1).
$$

For $\lambda=1$, one possible basis is

$$
\mathbf v_1=(1,-1,0),
\qquad
\mathbf v_2=(1,0,-1).
$$

These vectors are independent but not orthogonal because $\mathbf v_1\cdot\mathbf v_2=1$.

Apply Gram-Schmidt within the eigenspace:

$$
\mathbf w_1=\mathbf v_1=(1,-1,0).
$$

Next,

$$
\begin{aligned}
\mathbf w_2
&=\mathbf v_2-
\frac{\mathbf v_2\cdot\mathbf w_1}{\mathbf w_1\cdot\mathbf w_1}\mathbf w_1\\
&=(1,0,-1)-\frac12(1,-1,0)\\
&=\left(\frac12,\frac12,-1\right).
\end{aligned}
$$

Scale by $2$ to use the simpler vector

$$
\widetilde{\mathbf w}_2=(1,1,-2).
$$

Normalize:

$$
\mathbf q_1=\frac1{\sqrt2}(1,-1,0),
\qquad
\mathbf q_2=\frac1{\sqrt6}(1,1,-2).
$$

Then $\{\mathbf q_1,\mathbf q_2,\mathbf q_3\}$ is an orthonormal eigenbasis, and

$$
P^TCP=\operatorname{diag}(1,1,4).
$$

In [ ]:
C=sp.Matrix([[2,1,1],[1,2,1],[1,1,2]])
q1=sp.Matrix([1,-1,0])/sp.sqrt(2)
q2=sp.Matrix([1,1,-2])/sp.sqrt(6)
q3=sp.Matrix([1,1,1])/sp.sqrt(3)
P3=sp.Matrix.hstack(q1,q2,q3)
show_matrix(P3.T*P3,'P^TP')
show_matrix(sp.simplify(P3.T*C*P3),'P^TCP')

## Definitions and terminology

| Term | Meaning |
|---|---|
| Symmetric matrix | $A^T=A$. |
| Orthogonal matrix | $P^TP=I$, so $P^{-1}=P^T$. |
| Orthogonal diagonalization | $P^TAP=D$ with $P$ orthogonal and $D$ diagonal. |
| Spectral theorem | Real symmetric matrices have orthonormal eigenbases. |
| Spectral decomposition | $A=PDP^T$ or $A=\sum\lambda_i\mathbf q_i\mathbf q_i^T$. |
| Repeated eigenspace | Eigenspace associated with an eigenvalue of multiplicity greater than one. |

## Section 7.3 worksheet

1. Determine whether $\begin{bmatrix}5&2\\2&2\end{bmatrix}$ is symmetric.
2. Prove that if the columns of $P$ are orthonormal, then $P^{-1}=P^T$.
3. Orthogonally diagonalize $A=\begin{bmatrix}5&2\\2&2\end{bmatrix}$.
4. Verify directly that the eigenvectors $(2,1)$ and $(-1,2)$ are orthogonal.
5. For $C=\begin{bmatrix}2&1&1\\1&2&1\\1&1&2\end{bmatrix}$, verify that $(1,1,1)$ is an eigenvector and state its eigenvalue.
6. Apply one Gram-Schmidt step to $(1,-1,0)$ and $(1,0,-1)$.
7. Prove that an orthogonally diagonalizable real matrix is symmetric.

# Section 7.4 - Applications of Eigenvalues and Eigenvectors

## Learning objectives

Students should be able to:

- construct and analyze an age-transition population model;
- find and interpret a stable age distribution;
- solve a diagonalizable system $\mathbf y'=A\mathbf y$;
- represent a quadratic form by a symmetric matrix;
- use the principal-axes theorem to remove cross-product terms;
- classify quadratic forms using eigenvalues;
- prove and apply the constrained-extrema theorem.

## Population growth and stable age distributions

Consider a fictional wildlife population near a hiking preserve. Divide the population into three age classes:

1. first-year,
2. second-year,
3. third-year.

Use the Leslie matrix

$$
L=
\begin{bmatrix}
0&\frac65&2\\
\frac{11}{20}&0&0\\
0&\frac{11}{20}&0
\end{bmatrix}.
$$

The first row contains fertility rates. The subdiagonal contains survival rates.

A stable age distribution is an eigenvector with nonnegative entries. Verify that

$$
\mathbf x=
\begin{bmatrix}20\\10\\5\end{bmatrix}
$$

satisfies

$$
L\mathbf x
=
\begin{bmatrix}
22\\11\\11/2
\end{bmatrix}
=\frac{11}{10}
\begin{bmatrix}20\\10\\5\end{bmatrix}.
$$

Thus the growth factor is

$$
\lambda=\frac{11}{10}=1.1,
$$

and the age ratio remains

$$
20:10:5=4:2:1.
$$

The corresponding percentages are

$$
\frac47,\qquad\frac27,\qquad\frac17.
$$

In [ ]:
L=sp.Matrix([[0,sp.Rational(6,5),2],[sp.Rational(11,20),0,0],[0,sp.Rational(11,20),0]])
x0=sp.Matrix([20,10,5])
show_vector(L*x0,r'L\mathbf x')
show_exact(sp.factor(L.charpoly().as_expr()),r'p(\lambda)')
# Iterate a nearby initial distribution.
state=np.array([18.,12.,6.])
history=[state.copy()]
Lf=np.array(L.tolist(),dtype=float)
for _ in range(12):
    state=Lf@state
    history.append(state.copy())
plot_population(history,['first-year','second-year','third-year'])

> **Interpretation**
>
> The eigenvalue gives the long-run growth factor per transition, while the normalized positive eigenvector gives the long-run proportion in each age class.

## Systems of linear differential equations

Consider the coupled system

$$
\mathbf y'=A\mathbf y,
\qquad
A=
\begin{bmatrix}
-3&1\\
1&-3
\end{bmatrix}.
$$

This may be viewed as a fictional two-component badminton recovery model: each component decays but is weakly coupled to the other.

The eigenvalues are

$$
\lambda_1=-2,
\qquad
\lambda_2=-4,
$$

with eigenvectors

$$
\mathbf p_1=(1,1),
\qquad
\mathbf p_2=(1,-1).
$$

Form

$$
P=
\begin{bmatrix}
1&1\\
1&-1
\end{bmatrix},
\qquad
D=
\begin{bmatrix}
-2&0\\
0&-4
\end{bmatrix}.
$$

Set

$$
\mathbf y=P\mathbf w.
$$

Then

$$
P\mathbf w'=AP\mathbf w.
$$

Multiplying by $P^{-1}$ gives

$$
\mathbf w'=P^{-1}AP\mathbf w=D\mathbf w.
$$

Therefore the coupled system separates into

$$
w_1'=-2w_1,
\qquad
w_2'=-4w_2.
$$

Thus

$$
w_1=c_1e^{-2t},
\qquad
w_2=c_2e^{-4t}.
$$

Returning to the original variables,

$$
\mathbf y(t)
=c_1e^{-2t}
\begin{bmatrix}1\\1\end{bmatrix}
+c_2e^{-4t}
\begin{bmatrix}1\\-1\end{bmatrix}.
$$

If

$$
\mathbf y(0)=
\begin{bmatrix}3\\1\end{bmatrix},
$$

then

$$
\begin{bmatrix}3\\1\end{bmatrix}
=c_1
\begin{bmatrix}1\\1\end{bmatrix}
+c_2
\begin{bmatrix}1\\-1\end{bmatrix}.
$$

The equations are

$$
c_1+c_2=3,
\qquad
c_1-c_2=1.
$$

Adding gives $2c_1=4$, so $c_1=2$ and $c_2=1$. Hence

$$
\boxed{
\mathbf y(t)
=2e^{-2t}
\begin{bmatrix}1\\1\end{bmatrix}
+e^{-4t}
\begin{bmatrix}1\\-1\end{bmatrix}
}.
$$

In [ ]:
t=np.linspace(0,3,300)
y1=2*np.exp(-2*t)+np.exp(-4*t)
y2=2*np.exp(-2*t)-np.exp(-4*t)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(t,y1,label='component 1'); ax.plot(t,y2,label='component 2')
ax.grid(alpha=.3); ax.legend(); ax.set_xlabel('time'); ax.set_ylabel('state')
ax.set_title('Diagonalized differential-equation solution'); plt.show()

### Definition: quadratic form

> A quadratic form in $n$ variables is $q(\mathbf x)=\mathbf x^TA\mathbf x$, where $A$ may be taken symmetric.

For two variables,

$$
q(x,y)=ax^2+bxy+cy^2
$$

has symmetric matrix

$$
A=
\begin{bmatrix}
a&b/2\\
b/2&c
\end{bmatrix},
$$

because

$$
\begin{aligned}
\begin{bmatrix}x&y\end{bmatrix}
\begin{bmatrix}a&b/2\\b/2&c\end{bmatrix}
\begin{bmatrix}x\\y\end{bmatrix}
&=ax^2+\frac b2xy+\frac b2yx+cy^2\\
&=ax^2+bxy+cy^2.
\end{aligned}
$$

### Principal-axes theorem

> If $A$ is symmetric, choose an orthogonal matrix $P$ such that $P^TAP=D$. Under the rotation $\mathbf x=P\mathbf u$, the quadratic form becomes $q=\mathbf u^TD\mathbf u$, so all cross terms disappear.

### Proof

Let

$$
q(\mathbf x)=\mathbf x^TA\mathbf x
$$

and use the orthogonal coordinate change

$$
\mathbf x=P\mathbf u.
$$

Then

$$
\begin{aligned}
q
&=(P\mathbf u)^TA(P\mathbf u)\\
&=\mathbf u^TP^TAP\mathbf u\\
&=\mathbf u^TD\mathbf u.
\end{aligned}
$$

Because $D$ is diagonal,

$$
q=\lambda_1u_1^2+\cdots+\lambda_nu_n^2.
$$

No mixed products $u_iu_j$ remain. The new coordinate axes are precisely the orthonormal eigenvector directions. $\square$

## Detailed rotation of a conic

Consider

$$
5x^2+4xy+2y^2=12.
$$

The quadratic-form matrix is

$$
A=
\begin{bmatrix}
5&2\\
2&2
\end{bmatrix}.
$$

From Section 7.3, an orthogonal diagonalization is

$$
P=
\frac1{\sqrt5}
\begin{bmatrix}
2&-1\\
1&2
\end{bmatrix},
\qquad
P^TAP=
\begin{bmatrix}
6&0\\
0&1
\end{bmatrix}.
$$

Set

$$
\begin{bmatrix}x\\y\end{bmatrix}
=P
\begin{bmatrix}u\\v\end{bmatrix}.
$$

Then the equation becomes

$$
6u^2+v^2=12,
$$

or

$$
\frac{u^2}{2}+\frac{v^2}{12}=1.
$$

This is an ellipse whose principal axes are the eigenvector directions. The rotation angle satisfies

$$
\cos\theta=\frac2{\sqrt5},
\qquad
\sin\theta=\frac1{\sqrt5},
$$

so

$$
\theta=\arctan\left(\frac12\right).
$$

In [ ]:
plot_quadratic_contours([[5,2],[2,2]],12,'Rotated ellipse and principal axes')

## Classification by eigenvalue signs

For $q(\mathbf x)=\mathbf x^TA\mathbf x$ with symmetric $A$:

- all eigenvalues positive: $q$ is positive definite;
- all eigenvalues negative: $q$ is negative definite;
- both positive and negative eigenvalues: $q$ is indefinite;
- zero eigenvalues with no sign change: $q$ is semidefinite.

For example,

$$
4x^2+y^2+9z^2=36
$$

has three positive eigenvalues and describes an ellipsoid. In contrast,

$$
x^2+y^2-z^2=1
$$

has eigenvalues $1,1,-1$ and describes a one-sheet hyperboloid.

### Constrained-extrema theorem

> For a symmetric matrix $A$, the maximum of $q(\mathbf x)=\mathbf x^TA\mathbf x$ subject to $\|\mathbf x\|=1$ is the largest eigenvalue; the minimum is the smallest eigenvalue.

### Complete proof using the spectral theorem

Let

$$
A=PDP^T,
$$

where $P$ is orthogonal and

$$
D=\operatorname{diag}(\lambda_1,\ldots,\lambda_n).
$$

Set

$$
\mathbf y=P^T\mathbf x.
$$

Because $P$ is orthogonal,

$$
\|\mathbf y\|=\|\mathbf x\|=1.
$$

Then

$$
\begin{aligned}
q(\mathbf x)
&=\mathbf x^TA\mathbf x\\
&=\mathbf x^TPDP^T\mathbf x\\
&=\mathbf y^TD\mathbf y\\
&=\lambda_1y_1^2+\cdots+\lambda_ny_n^2.
\end{aligned}
$$

Since

$$
y_1^2+\cdots+y_n^2=1,
$$

the value of $q$ is a weighted average of the eigenvalues, with nonnegative weights $y_i^2$. Therefore,

$$
\lambda_{\min}\le q(\mathbf x)\le\lambda_{\max}.
$$

The maximum is achieved when $\mathbf y$ is a coordinate vector corresponding to $\lambda_{\max}$, which means $\mathbf x$ is a unit eigenvector for $\lambda_{\max}$. Similarly, the minimum occurs at a unit eigenvector for $\lambda_{\min}$. $\square$

For

$$
q(x,y)=5x^2+4xy+2y^2,
\qquad
x^2+y^2=1,
$$

the eigenvalues are $6$ and $1$. Therefore,

$$
\max q=6
$$

at

$$
\pm\frac1{\sqrt5}(2,1),
$$

and

$$
\min q=1
$$

at

$$
\pm\frac1{\sqrt5}(-1,2).
$$

### Rayleigh quotient

For nonzero $\mathbf x$,

$$
R_A(\mathbf x)=\frac{\mathbf x^TA\mathbf x}{\mathbf x^T\mathbf x}.
$$

If $\mathbf x$ is an eigenvector with eigenvalue $\lambda$, then $R_A(\mathbf x)=\lambda$. For an approximate eigenvector it provides an eigenvalue estimate.


In [ ]:
theta=np.linspace(0,2*np.pi,500)
x=np.cos(theta); y=np.sin(theta)
q=5*x**2+4*x*y+2*y**2
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(theta,q)
ax.axhline(6,linestyle='--',label='maximum eigenvalue 6')
ax.axhline(1,linestyle='--',label='minimum eigenvalue 1')
ax.set_xlabel('angle on unit circle'); ax.set_ylabel('quadratic-form value')
ax.grid(alpha=.3); ax.legend(); ax.set_title('Rayleigh quotient on the unit circle'); plt.show()

## Definitions and terminology

| Term | Meaning |
|---|---|
| Leslie matrix | Age-transition matrix containing fertility and survival rates. |
| Stable age distribution | Positive eigenvector giving constant long-run proportions. |
| Growth factor | Dominant positive eigenvalue of the population matrix. |
| Linear system $\mathbf y'=A\mathbf y$ | Coupled first-order differential equations. |
| Quadratic form | $q(\mathbf x)=\mathbf x^TA\mathbf x$. |
| Principal axes | Orthogonal eigenvector directions of a symmetric quadratic-form matrix. |
| Positive definite | All eigenvalues positive. |
| Rayleigh quotient | $\mathbf x^TA\mathbf x/(\mathbf x^T\mathbf x)$. |

## Section 7.4 worksheet

1. Verify that $(20,10,5)^T$ is an eigenvector of the Leslie matrix in the notes and find its eigenvalue.
2. Convert the stable age vector in Problem 1 to percentages.
3. Solve $\mathbf y'=\begin{bmatrix}-3&1\\1&-3\end{bmatrix}\mathbf y$ when $\mathbf y(0)=(3,1)^T$.
4. Find the symmetric matrix of $q(x,y)=5x^2+4xy+2y^2$.
5. Use the orthogonal matrix in the notes to transform $5x^2+4xy+2y^2=12$ into principal-axis coordinates.
6. Classify $q(x,y)=x^2-4y^2$ using eigenvalue signs.
7. Find the maximum and minimum of $5x^2+4xy+2y^2$ subject to $x^2+y^2=1$.
8. Prove that the orthogonal change of coordinates preserves vector length.

# Chapter 7 summary

The central idea of Chapter 7 is that a matrix may possess special directions that it only scales. These eigendirections reveal the long-term behavior of repeated transformations, permit diagonalization, explain the geometry of symmetric matrices, decouple differential equations, determine stable population structures, remove cross terms from quadratic forms, and solve constrained optimization problems.

A useful conceptual chain is

$$
\text{eigenvectors}
\Longrightarrow
\text{eigenbasis}
\Longrightarrow
\text{diagonalization}
\Longrightarrow
\text{simple powers and applications}.
$$

For symmetric matrices, the spectral theorem strengthens this chain by guaranteeing an **orthonormal** eigenbasis.